
# Polymer Architectures & Resistance-Fluctuation Duality in Rouse Dynamics

This example bridges :class:`~pylimer_tools_cpp.RouseBrownianSimulator` and
:class:`~pylimer_tools_cpp.ResistanceDistanceAnalysis`, illustrating how
topological effective resistances govern the equilibrium conformational
fluctuations of Gaussian polymer architectures and crosslinked networks.

For any connected spring network with scalar Laplacian $\mathbf{L}$,
spring constant $k$, and temperature $T$:

\begin{align}\langle |\mathbf{r}_i - \mathbf{r}_j|^2 \rangle_{\mathrm{eq}}
   = \frac{3 k_B T}{k} R_{ij},
   \qquad
   \langle R_g^2 \rangle_{\mathrm{eq}}
   = \frac{3 k_B T}{k N^2} K(G),
   \qquad
   \langle |\mathbf{r}_i - \mathbf{r}_{\mathrm{CM}}|^2 \rangle_{\mathrm{eq}}
   = \frac{3 k_B T}{k} L^+_{ii}
   = \frac{3 k_B T}{k N}\left(S_i - \frac{K(G)}{N}\right),\end{align}

where $R_{ij}$ is the effective resistance, $K(G)$ is the
Kirchhoff index, $\mathbf{L}^+$ is the Moore-Penrose pseudo-inverse,
and $S_i = \sum_j R_{ij}$ is the total resistance per node.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pylimer_tools_cpp import (
    ResistanceDistanceAnalysis,
    RouseBrownianSimulator,
    Universe,
)

## Rouse Dynamics Across Polymer Architectures (Linear, Ring, Star, Complete & Weighted)
We instantiate :class:`~pylimer_tools_cpp.RouseBrownianSimulator` across
canonical topologies
(:meth:`~pylimer_tools_cpp.RouseBrownianSimulator.create_linear_chain`,
:meth:`~pylimer_tools_cpp.RouseBrownianSimulator.create_ring`,
:meth:`~pylimer_tools_cpp.RouseBrownianSimulator.create_star`,
:meth:`~pylimer_tools_cpp.RouseBrownianSimulator.create_complete`) as well as
:meth:`~pylimer_tools_cpp.RouseBrownianSimulator.from_matrix` and custom
weighted edges, using $N = 13$ beads ($f = 4$ star arms of length
$M = 3$). Despite differing numbers of bonds, equipartition predicts
the same mean harmonic potential energy
$\langle U \rangle_{\mathrm{eq}} = \frac{3}{2} (N - 1) k_B T$
for every connected $N$-bead Gaussian network.



In [ ]:
n_nodes = 13  # 1 + 4 * 3 = 13 nodes so all architectures share the same N
num_arms, arm_len = 4, 3
gamma, k, temp, dt = 1.0, 1.0, 1.0, 0.01

# Also build a heterogeneous weighted cyclic network and a from_matrix twin
weighted_from = list(range(n_nodes - 1)) + [n_nodes - 1, 0, 3]
weighted_to = list(range(1, n_nodes)) + [0, 6, 9]
weighted_k = [1.0] * n_nodes + [2.5, 2.5]

arch_specs = [
    (
        "Linear chain",
        RouseBrownianSimulator.create_linear_chain(n_nodes, gamma, k, temp, dt),
        ResistanceDistanceAnalysis.create_linear_chain(n_nodes),
        0,
        n_nodes - 1,
    ),
    (
        "Ring polymer",
        RouseBrownianSimulator.create_ring(n_nodes, gamma, k, temp, dt),
        ResistanceDistanceAnalysis.create_ring(n_nodes),
        0,
        n_nodes // 2,
    ),
    (
        "Star (4x3)",
        RouseBrownianSimulator.create_star(num_arms, arm_len, gamma, k, temp, dt),
        ResistanceDistanceAnalysis.create_star(num_arms, arm_len),
        arm_len,
        2 * arm_len,
    ),
    (
        "Complete K_N",
        RouseBrownianSimulator.create_complete(n_nodes, gamma, k, temp, dt),
        ResistanceDistanceAnalysis.create_complete(n_nodes),
        0,
        n_nodes - 1,
    ),
    (
        "Weighted bridged ring",
        RouseBrownianSimulator(
            weighted_from,
            weighted_to,
            gamma=gamma,
            k=k,
            T=temp,
            dt=dt,
            num_nodes=n_nodes,
            weights=weighted_k,
        ),
        ResistanceDistanceAnalysis(
            weighted_from, weighted_to, weights=weighted_k, num_nodes=n_nodes
        ),
        0,
        n_nodes // 2,
    ),
]

# Verify from_matrix produces identical Laplacian and dynamics
sim_from_mat = RouseBrownianSimulator.from_matrix(
    arch_specs[0][1].get_laplacian(), gamma=gamma, k=k, T=temp, dt=dt
)
assert np.allclose(
    sim_from_mat.get_laplacian().toarray(),
    arch_specs[0][1].get_laplacian().toarray(),
)

n_eq_steps = 4000
n_prod_steps = 10000
sample_interval = 20  # dt_sample = 0.2 -> 501 frames per trajectory
n_repeats = 16
u_equipartition = 1.5 * (n_nodes - 1) * temp

arch_msd: dict[str, np.ndarray] = {}
arch_pair_dist_sq: dict[str, np.ndarray] = {}
summary_rows = []
times_arch = None

for name, sim, rda, node_a, node_b in arch_specs:
    msd_reps = []
    rg_sq_samples = []
    ree_sq_samples = []
    u_samples = []
    r0_sq_accum = np.zeros(n_nodes)
    n_frames_total = 0
    l_dense = rda.get_laplacian().toarray()

    for rep in range(n_repeats):
        sim.set_seed(3000 + rep)
        sim.initialize_random_walk(bond_length=np.sqrt(3.0 * temp / k), seed=700 + rep)
        sim.step(n_eq_steps)

        res = sim.run(
            total_steps=n_prod_steps,
            sample_interval=sample_interval,
            save_trajectory=True,
            end_to_end_node_a=node_a,
            end_to_end_node_b=node_b,
        )
        if times_arch is None:
            times_arch = res.times
        msd_reps.append(
            RouseBrownianSimulator.compute_msd(res.trajectory, time_averaged=True)
        )

        # Accumulate equilibrium conformational observables over trajectory frames
        traj = np.asarray(res.trajectory)
        com = traj.mean(axis=1, keepdims=True)
        rg_sq_samples.append(np.mean(np.sum((traj - com) ** 2, axis=2), axis=1))
        ree_sq_samples.append(
            np.sum((traj[:, node_b, :] - traj[:, node_a, :]) ** 2, axis=1)
        )
        u_samples.append(0.5 * k * np.sum(traj * (l_dense @ traj), axis=(1, 2)))
        r0_sq_accum += np.sum((traj - traj[:, 0:1, :]) ** 2, axis=(0, 2))
        n_frames_total += traj.shape[0]

    arch_msd[name] = np.mean(msd_reps, axis=0)
    arch_pair_dist_sq[name] = r0_sq_accum / float(n_frames_total)

    # Exact resistance-based theoretical predictions
    prefactor = 3.0 * temp / k
    kirchhoff = rda.compute_kirchhoff_index()
    rg_sq_theory = prefactor * kirchhoff / float(n_nodes**2)
    ree_sq_theory = prefactor * rda.compute_effective_resistance(node_a, node_b)

    summary_rows.append(
        {
            "Architecture": name,
            "K(G)": f"{kirchhoff:.2f}",
            "<Rg^2> [Sim]": f"{np.mean(rg_sq_samples):.3f}",
            "<Rg^2> [Theory]": f"{rg_sq_theory:.3f}",
            "<Ree^2> [Sim]": f"{np.mean(ree_sq_samples):.3f}",
            "<Ree^2> [Theory]": f"{ree_sq_theory:.3f}",
            "<U> [Sim]": f"{np.mean(u_samples):.2f}",
            "<U> [Equipart.]": f"{u_equipartition:.2f}",
        }
    )

## Fluctuation-Resistance Theorem Verification
The summary table confirms that
:meth:`~pylimer_tools_cpp.ResistanceDistanceAnalysis.compute_kirchhoff_index`
and
:meth:`~pylimer_tools_cpp.ResistanceDistanceAnalysis.compute_effective_resistance`
predict the simulated equilibrium $\langle R_g^2 \rangle$ and
$\langle R_{ij}^2 \rangle$ across every architecture without fitting
parameters.



In [ ]:
print(pd.DataFrame(summary_rows).to_string(index=False))

## Thermal Fluctuations & Resistance Centrality in a Crosslinked Polymer Network
We construct a connected crosslinked polymer network :class:`~pylimer_tools_cpp.Universe`
containing 4-functional crosslinks, bridging strands, cyclic loops, and two
dangling chain ends, then instantiate both
:meth:`~pylimer_tools_cpp.RouseBrownianSimulator.from_universe` and
:meth:`~pylimer_tools_cpp.ResistanceDistanceAnalysis.from_universe`.
Around the network center of mass, node $i$'s mean-squared positional
fluctuation obeys

\begin{align}\langle |\mathbf{r}_i - \mathbf{r}_{\mathrm{CM}}|^2 \rangle_{\mathrm{eq}}
   = \frac{3 k_B T}{k} L^+_{ii}
   = \frac{3 k_B T}{k N} \left( S_i - \frac{K(G)}{N} \right),\end{align}

showing quantitatively why topologically central crosslinkers fluctuate far
less than dangling chain ends.



In [ ]:
def _build_crosslinked_universe() -> tuple[Universe, list[int], list[int], list[int]]:
    """Build a small connected crosslinked polymer network with loops and dangling ends."""
    universe = Universe(20.0, 20.0, 20.0)
    # 4 tetra-functional crosslinks (atom IDs 1..4) connected by 6 bridging
    # strands of 4 beads each (24 strand beads) plus 2 dangling tails of 5 beads
    # (10 dangling beads) -> 38 atoms total.
    xl_ids = [1, 2, 3, 4]
    bonds_from: list[int] = []
    bonds_to: list[int] = []
    next_id = 5
    bridging_beads: list[int] = []
    dangling_beads: list[int] = []

    bridge_pairs = [(1, 2), (2, 3), (3, 4), (4, 1), (1, 3), (2, 4)]
    for u, v in bridge_pairs:
        prev = u
        for _ in range(4):
            bridging_beads.append(next_id)
            bonds_from.append(prev)
            bonds_to.append(next_id)
            prev = next_id
            next_id += 1
        bonds_from.append(prev)
        bonds_to.append(v)

    for anchor in (1, 3):
        prev = anchor
        for _ in range(5):
            dangling_beads.append(next_id)
            bonds_from.append(prev)
            bonds_to.append(next_id)
            prev = next_id
            next_id += 1

    total_atoms = next_id - 1
    atom_ids = list(range(1, total_atoms + 1))
    atom_types = [2 if aid in xl_ids else 1 for aid in atom_ids]

    rng = np.random.default_rng(123)
    coords = rng.normal(0.0, 1.5, size=(total_atoms, 3))
    zeros_i = [0] * total_atoms
    universe.add_atoms(
        atom_ids,
        atom_types,
        coords[:, 0].tolist(),
        coords[:, 1].tolist(),
        coords[:, 2].tolist(),
        zeros_i,
        zeros_i,
        zeros_i,
    )
    universe.add_bonds(bonds_from, bonds_to)
    return universe, xl_ids, bridging_beads, dangling_beads


net_u, xl_atom_ids, bridge_atom_ids, dangling_atom_ids = _build_crosslinked_universe()
net_rda = ResistanceDistanceAnalysis.from_universe(net_u)
assert net_rda.is_connected()
n_net = net_rda.get_nr_of_nodes()

# Map atom IDs to 0-based vertex indices matching ResistanceDistanceAnalysis
xl_idx = [net_u.get_vertex_idx_by_atom_id(aid) for aid in xl_atom_ids]
bridge_idx = [net_u.get_vertex_idx_by_atom_id(aid) for aid in bridge_atom_ids]
dangling_idx = [net_u.get_vertex_idx_by_atom_id(aid) for aid in dangling_atom_ids]

# Theoretical positional fluctuation amplitude from L^+ and S_i
l_plus_net = net_rda.compute_moore_penrose_pseudo_inverse()
tot_res_net = net_rda.compute_total_resistance_per_node()
kirchhoff_net = net_rda.compute_kirchhoff_index()
centrality_net = net_rda.compute_resistance_centrality()

prefactor_net = 3.0 * temp / k
fluct_theory_lplus = prefactor_net * np.diag(l_plus_net)
fluct_theory_si = (
    prefactor_net * (tot_res_net - kirchhoff_net / float(n_net)) / float(n_net)
)
assert np.allclose(fluct_theory_lplus, fluct_theory_si, atol=1e-10)

# Equilibrate and sample network Brownian dynamics via from_universe
net_fluct_accum = np.zeros(n_net)
net_samples = 0
n_net_reps = 10
for rep in range(n_net_reps):
    net_sim = RouseBrownianSimulator.from_universe(
        net_u, gamma=gamma, k=k, T=temp, dt=0.01
    )
    net_sim.set_seed(9000 + rep)
    net_sim.step(6000)
    res_net = net_sim.run(total_steps=8000, sample_interval=20, save_trajectory=True)
    for frame in res_net.trajectory:
        cm = frame.mean(axis=0, keepdims=True)
        net_fluct_accum += np.sum((frame - cm) ** 2, axis=1)
        net_samples += 1

fluct_sim_net = net_fluct_accum / float(net_samples)
rel_err_net = (
    np.mean(np.abs(fluct_sim_net - fluct_theory_lplus) / fluct_theory_lplus) * 100.0
)
print(
    f"\nCrosslinked network (N={n_net}, edges={net_rda.get_nr_of_edges()}): "
    f"mean relative error of <|r_i - r_CM|^2> vs (3 kB T / k) L^+_ii = {rel_err_net:.2f}%"
)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15.5, 4.7))

# Panel 1: Monomer MSD across polymer architectures
assert times_arch is not None
arch_colors = {
    "Linear chain": "#1f77b4",
    "Ring polymer": "#ff7f0e",
    "Star (4x3)": "#2ca02c",
    "Complete K_N": "#d62728",
    "Weighted bridged ring": "#9467bd",
}
pos_mask = times_arch > 0
for name, _, _, _, _ in arch_specs:
    ax1.loglog(
        times_arch[pos_mask],
        arch_msd[name][pos_mask],
        "-",
        color=arch_colors[name],
        linewidth=2.0,
        label=name,
    )
ax1.loglog(
    times_arch[pos_mask],
    6.0 * (temp / gamma / float(n_nodes)) * times_arch[pos_mask],
    "k--",
    linewidth=1.5,
    label=r"CM asymptote $6(D/N)\tau$",
)
ax1.set_xlabel(r"Lag Time $\tau$")
ax1.set_ylabel(r"Monomer MSD $\langle \Delta r^2(\tau) \rangle$")
ax1.set_title("Monomer Dynamics Across Topologies")
ax1.grid(True, which="both", linestyle="--", alpha=0.45)
ax1.legend(frameon=True, fontsize=8.0, loc="lower right")

# Panel 2: Pairwise <r_{0j}^2> vs (3 kB T / k) R_{0j}
j_idx = np.arange(n_nodes)
for name, _, rda, _, _ in arch_specs[:3]:
    c = arch_colors[name]
    r0j_theory = np.array(
        [(3.0 * temp / k) * rda.compute_effective_resistance(0, int(j)) for j in j_idx]
    )
    ax2.plot(
        j_idx, r0j_theory, "-", color=c, linewidth=2.0, label=f"{name} (resistance)"
    )
    ax2.plot(
        j_idx,
        arch_pair_dist_sq[name],
        "o",
        color=c,
        markersize=5,
        label=f"{name} (Brownian sim.)",
    )

ax2.set_xlabel("Node index $j$")
ax2.set_ylabel(
    r"Mean-Squared Distance $\langle |\mathbf{r}_0 - \mathbf{r}_j|^2 \rangle$"
)
ax2.set_title("Fluctuation-Resistance Duality")
ax2.grid(True, linestyle="--", alpha=0.45)
ax2.legend(frameon=True, fontsize=7.5, ncol=2)

# Panel 3: Network bead fluctuations vs resistance centrality
sort_order = np.argsort(centrality_net)
ax3.plot(
    centrality_net[sort_order],
    fluct_theory_lplus[sort_order],
    "k-",
    linewidth=1.8,
    label=r"Exact $\frac{3 k_B T}{k} L^+_{ii}$",
)
for role_label, idxs, marker, col in (
    ("Crosslinkers", xl_idx, "D", "#d62728"),
    ("Bridging beads", bridge_idx, "o", "#1f77b4"),
    ("Dangling ends", dangling_idx, "^", "#2ca02c"),
):
    ax3.scatter(
        centrality_net[idxs],
        fluct_sim_net[idxs],
        marker=marker,
        s=42,
        color=col,
        zorder=4,
        label=f"{role_label} (sim.)",
    )

ax3.set_xlabel("Resistance Centrality $C_R(i)$")
ax3.set_ylabel(
    r"Bead Fluctuation $\langle |\mathbf{r}_i - \mathbf{r}_{\mathrm{CM}}|^2 \rangle$"
)
ax3.set_title("Network Fluctuations vs. Centrality")
ax3.grid(True, linestyle="--", alpha=0.45)
ax3.legend(frameon=True, fontsize=8.0, loc="upper right")

plt.tight_layout()
plt.show()